---
sidebar_label: LangGraph Store
---


# LangGraph Store with OceanBase

This notebook demonstrates `OceanBaseStore` as the LangGraph long-term memory backend.

It covers:
- creating a store with deterministic demo embeddings
- writing namespace-scoped items
- exact lookup and semantic search
- listing namespaces for multi-tenant memory layouts


## Installation

Install the package together with LangGraph support:


In [ ]:
%pip install -qU langchain-oceanbase langgraph

## Start OceanBase

For a local demo, start OceanBase in Docker:


In [ ]:
!docker run --name=oceanbase -e MODE=mini -e OB_SERVER_IP=127.0.0.1 -p 2881:2881 -d oceanbase/oceanbase-ce:latest

## Define lightweight demo embeddings

The notebook uses a small deterministic embedding implementation so it works without the optional `pyseekdb` extra.


In [ ]:
from langchain_core.embeddings import Embeddings


class DemoEmbeddings(Embeddings):
    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed(text) for text in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._embed(text)

    async def aembed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.embed_documents(texts)

    async def aembed_query(self, text: str) -> list[float]:
        return self.embed_query(text)

    def _embed(self, text: str) -> list[float]:
        lowered = text.lower()
        return [
            1.0 if "python" in lowered else 0.0,
            1.0 if "database" in lowered else 0.0,
            float((len(lowered) % 13) + 1),
        ]

## Create and initialize the store


In [ ]:
from langchain_oceanbase import OceanBaseStore


store = OceanBaseStore(
    connection_args={
        "host": "127.0.0.1",
        "port": "2881",
        "user": "root@test",
        "password": "",
        "db_name": "test",
    },
    index={"dims": 3, "embed": DemoEmbeddings(), "fields": ["memory"]},
    ttl_config={"refresh_on_read": True, "default_ttl": 60},
)
store.setup()

## Write namespace-scoped memory


In [ ]:
namespace = ("memories", "user-123")

store.put(
    namespace,
    "favorite-language",
    {"memory": "The user prefers Python for data tooling."},
)
store.put(
    namespace,
    "database-note",
    {"memory": "OceanBase is the preferred database backend."},
    ttl=30,
)

## Exact lookup


In [ ]:
exact_item = store.get(namespace, "favorite-language")
exact_item

## Semantic search


In [ ]:
semantic_results = store.search(namespace, query="python tooling", limit=2)
semantic_results

## Namespace discovery

This is useful when you organize memories by user, tenant, or workflow namespace prefixes.


In [ ]:
store.list_namespaces(prefix=("memories",))

## Next Steps

- See [examples/langgraph_store.py](../examples/langgraph_store.py) for the full runnable script.
- Use `ttl=` and `ttl_config=` when you want expiring long-term memory.
- Pair `OceanBaseStore` with `OceanBaseCheckpointSaver` when you want both graph state persistence and a separate long-term memory surface.
